# ミミトレ (eng-listening) コンテンツパイプライン — Google Colab GPU 版

レッスン生成 (Cloud TTS) と単語アライメント (faster-whisper) を Colab の GPU ランタイムで実行するためのノートブックです。Mac mini 上のローカル実行 (cpu / int8) と同じパイプライン (`tools/content/`) を、GPU (cuda / float16) で高速に回します。

## 前提

- ランタイムが **GPU** であること (ランタイム > ランタイムのタイプを変更 > GPU)。T4 が最も安く、この用途には十分です。
- **takezou621@gmail.com** で Colab にサインインしていること (TTS の認証 ADC とクォータプロジェクトの都合)。
- 対象ブランチに Colab 用シム (cloud_tts.py の ADC フォールバック、build_lesson.py の whisper 環境変数上書き) が含まれていること。

## 警告 (必読)

- **声・設定の一貫性ルールは `tools/content/voice_config.json` が正** です。このノートブックや Colab 側で声・話速・モデルを場当たり的に変えないこと。
- **TTS は月 90 万文字 (monthlyStopChars) で自動停止** します (無料枠 100 万文字の手前)。再生成分も累計に含まれます。
- **usage/ 台帳と build/ は Drive 経由で Mac mini と共有** します。同期せずに両環境で生成すると台帳がずれ、二重に文字数を消費したり設定ハッシュの混在を招きます。**生成前に必ず Sync In、生成後に必ず Sync Out** を実行してください (このノートブックのセルがその順序になっています)。

In [ ]:
# ===== パラメータ (必要に応じて書き換える) =====
LESSON_ID = "phone-appointment"      # lessons/<id>.json のレッスン id
BRANCH = "main"       # Colab 用シムを含むブランチ
DRIVE_DIR = "eng-listening-content"  # Drive 上の共有フォルダ名 (MyDrive 直下)

REPO_DIR = "/content/eng-listening"
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_DIR}"
print("LESSON_ID:", LESSON_ID, "/ BRANCH:", BRANCH, "/ DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# ===== ランタイム確認 =====
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("ffmpeg:", shutil.which("ffmpeg"))

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("警告: GPU が見えません。ランタイム > ランタイムのタイプを変更 で GPU (T4 等) を選んでから再実行してください。")

In [ ]:
# ===== Google Drive マウント =====
from google.colab import drive

drive.mount("/content/drive")

import os

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive 共有フォルダ:", DRIVE_ROOT)

In [ ]:
# ===== リポジトリ取得 + Sync In (Drive -> リポジトリ) =====
import os
import shutil
import subprocess


def run(cmd, **kwargs):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)


if os.path.exists(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
else:
    run(["git", "clone", "--branch", BRANCH,
         "https://github.com/takezou621/eng-listening", REPO_DIR])

# Sync In: 台帳 (usage/) と生成途中物 (build/) を Drive から取り込む。
# これを飛ばすと月間文字数カウンタがずれ、二重生成の原因になる。
CONTENT_DIR = os.path.join(REPO_DIR, "tools", "content")
for name in ("usage", "build"):
    src = os.path.join(DRIVE_ROOT, name)
    dst = os.path.join(CONTENT_DIR, name)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"sync in: Drive の {name}/ を取り込みました")
    else:
        print(f"sync in: Drive に {name}/ はまだありません (初回は空で開始)")

In [ ]:
# ===== 依存インストール + CUDA チェック =====
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--project", "tools/content"], cwd=REPO_DIR, check=True)

# Colab の GPU 実行には CUDA 12 系の cuBLAS / cuDNN が要る (ctranslate2 は同梱しないため venv に追加する)
subprocess.run(["uv", "pip", "install", "--python", "tools/content/.venv/bin/python",
                "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], cwd=REPO_DIR, check=True)
import glob, os
_lib_dirs = []
for _pat in ("tools/content/.venv/lib/python*/site-packages/nvidia/cublas/lib",
             "tools/content/.venv/lib/python*/site-packages/nvidia/cudnn/lib"):
    _lib_dirs += glob.glob(os.path.join(str(REPO_DIR), _pat))
if _lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(_lib_dirs + ([os.environ["LD_LIBRARY_PATH"]] if os.environ.get("LD_LIBRARY_PATH") else []))
    print("CUDA lib dirs:", _lib_dirs)

# faster-whisper の実体 ctranslate2 から CUDA の見え方を確認する
# (uv sync はプロジェクトの venv に入れるため、カーネルから直接 import せず venv 経由で確認する)
_check = subprocess.run(
    ["uv", "run", "--project", "tools/content", "python", "-c",
     "import ctranslate2; print(ctranslate2.get_cuda_device_count())"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
CUDA_COUNT = int(_check.stdout.strip().splitlines()[-1]) if _check.returncode == 0 and _check.stdout.strip() else 0
print("CUDA devices (ctranslate2):", CUDA_COUNT)
if CUDA_COUNT == 0:
    print("警告: CUDA が使えません。アライメントは CPU (cpu / int8) にフォールバックします (遅くなります)。")

In [ ]:
# ===== 認証 (ADC) + 環境変数 =====
import json
import os
import urllib.request

from google.colab import auth

# takezou621@gmail.com でサインインした状態で実行すること
auth.authenticate_user()

# TTS: gcloud CLI が無いので ADC フォールバックを有効化 (cloud_tts.py のシム)
os.environ["EL_TTS_AUTH_FALLBACK"] = "adc"
# Whisper: GPU が見えれば cuda / float16、無ければ従来どおり cpu / int8
if CUDA_COUNT > 0:
    os.environ["EL_WHISPER_DEVICE"] = "cuda"
    os.environ["EL_WHISPER_COMPUTE_TYPE"] = "float16"
else:
    os.environ["EL_WHISPER_DEVICE"] = "cpu"
    os.environ["EL_WHISPER_COMPUTE_TYPE"] = "int8"
print("EL_WHISPER_DEVICE:", os.environ["EL_WHISPER_DEVICE"],
      "/ EL_WHISPER_COMPUTE_TYPE:", os.environ["EL_WHISPER_COMPUTE_TYPE"])

# ADC の動作確認 (アクセストークン自体は決して表示しない)
import google.auth
import google.auth.transport.requests

credentials, _project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())
req = urllib.request.Request(
    "https://openidconnect.googleapis.com/v1/userinfo",
    headers={"Authorization": f"Bearer {credentials.token}"},
)
with urllib.request.urlopen(req) as resp:
    info = json.loads(resp.read().decode("utf-8"))
print("ADC account:", info.get("email"))

with open(os.path.join(CONTENT_DIR, "voice_config.json"), encoding="utf-8") as f:
    voice_cfg = json.load(f)
print("voice_config gcloudAccount:", voice_cfg["gcloudAccount"],
      "/ quotaProject (X-Goog-User-Project):", voice_cfg["quotaProject"])
if info.get("email") != voice_cfg["gcloudAccount"]:
    print("警告: ADC のアカウントが voice_config の gcloudAccount と一致しません。TTS が 403 になる可能性があります。")

In [ ]:
# ===== Step 1: TTS 生成 (台本 JSON -> 行 WAV) =====
# 月間文字数カウンタ (90 万文字で自動停止) は generate_lesson.py / cloud_tts.py が管理する。
import subprocess

subprocess.run(
    ["uv", "run", "--project", "tools/content", "python",
     "tools/content/generate_lesson.py", "--lesson", LESSON_ID],
    cwd=REPO_DIR, check=True,
)
print("generate_lesson.py 完了")

In [ ]:
# ===== Step 2: 一貫性ゲート -> Step 3: ビルド (アライメント + lesson.json) =====
# ローカルと同じ順序: ゲート PASS でなければ build_lesson.py はビルドを拒否する。
# ゲートが落ちた行は削除して Step 1 を再実行すること (声の一貫性ルール)。
import subprocess

subprocess.run(
    ["uv", "run", "--project", "tools/content", "python",
     "tools/content/check_voice_consistency.py", "--lesson", LESSON_ID],
    cwd=REPO_DIR, check=True,
)
print("check_voice_consistency.py: PASS")

subprocess.run(
    ["uv", "run", "--project", "tools/content", "python",
     "tools/content/build_lesson.py", "--lesson", LESSON_ID],
    cwd=REPO_DIR, check=True,
)
print("build_lesson.py 完了")

In [ ]:
# ===== 出力検証 =====
import json
import os

path = os.path.join(REPO_DIR, "assets", "lessons", LESSON_ID, "lesson.json")
with open(path, encoding="utf-8") as f:
    data = json.load(f)

sentences = data["sentences"]
total_words = 0
for s in sentences:
    words = s.get("words", [])
    assert words, f"words が空の文があります: sentence {s.get('index')}"
    for w in words:
        assert w["startSec"] < w["endSec"], f"単語の時刻が不正です: {w}"
        total_words += 1
print(f"lesson.json OK: {len(sentences)} 文 / {total_words} 単語 (全単語 startSec < endSec)")

if LESSON_ID == "phone-appointment":
    for s in sentences:
        for w in s.get("words", []):
            if w["text"] == "Miller":
                dur = w["endSec"] - w["startSec"]
                print(f"Miller: start={w['startSec']} end={w['endSec']} duration={dur:.3f}s "
                      "(修正前は 0.060s。再アライメントで自然な長さになっていれば成功)")

In [ ]:
# ===== Sync Out (リポジトリ -> Drive) + 成果物 zip =====
import os
import shutil

# 台帳は必ず Drive に戻す (Mac mini 側と台帳を一致させる)
shutil.copytree(os.path.join(CONTENT_DIR, "usage"),
                os.path.join(DRIVE_ROOT, "usage"), dirs_exist_ok=True)
print("sync out: usage/")

build_dir = os.path.join(CONTENT_DIR, "build", LESSON_ID)
if os.path.isdir(build_dir):
    shutil.copytree(build_dir, os.path.join(DRIVE_ROOT, "build", LESSON_ID),
                    dirs_exist_ok=True)
    print("sync out: build/", LESSON_ID)

assets_dir = os.path.join(REPO_DIR, "assets", "lessons", LESSON_ID)
shutil.copytree(assets_dir,
                os.path.join(DRIVE_ROOT, "assets", "lessons", LESSON_ID),
                dirs_exist_ok=True)
print("sync out: assets/lessons/", LESSON_ID)

zip_path = shutil.make_archive(
    f"/content/{LESSON_ID}-assets", "zip",
    os.path.join(REPO_DIR, "assets", "lessons"), LESSON_ID,
)
print("zip:", zip_path)
# zip は /content に作るだけにする。自動ダウンロードはしない (実行者のブラウザに
# 繰り返し落ちるため)。必要な場合は Colab のファイルペインから手動でダウンロードする。
# 成果物の正本は Drive の eng-listening-content/ である。


## 結果の持ち帰り

1. 成果物は Drive の `eng-listening-content/` に保存されています (`usage/`、`build/<レッスンID>/`、`assets/lessons/<レッスンID>/`)。zip は /content に作られますが自動ダウンロードはされません（必要な場合はファイルペインから手動で取得）。
2. Mac mini 側で Drive から `assets/lessons/<レッスンID>/` をリポジトリに取り込み、差分を確認してから **PR 経由** で main に入れます (main への直接 push は禁止)。
3. `usage/` 台帳は Mac mini 側の `tools/content/usage/` にも反映してください (次に mini で生成するときのカウンタを一致させるため)。

## トラブルシューティング

- **CUDA が見えない** → ランタイム > ランタイムのタイプを変更 でハードウェア アクセラレータに GPU を選び、ランタイムを再起動して最初から実行する。
- **TTS が 403** → ADC のアカウントが takezou621@gmail.com か確認する (認証セルの出力)。別アカウントで authenticate_user するとクォータプロジェクト (muse-gemini-514814) への権限が無く 403 になる。X-Goog-User-Project ヘッダは voice_config.json の quotaProject から自動で付く。
- **一貫性ゲートが落ちる** → 外れ値の行 WAV を削除し、Step 1 から再実行する。声や設定を変えてごまかさない (voice_config.json が正)。
- **Computing Unit (CU) 消費について** → T4 が最も安い GPU です。アライメントはレッスン 1 本あたり数分程度の短い作業なので消費は小さく、Mac mini の CPU で数十分待つより Colab の方が速く済みます。長時間ランタイムを起動したまま放置すると消費が増えるので、終わったらランタイムを切断してください。